# Adapter and neuron attribution — 06a Colab rehearsal (bug-finding only)

**Do not run without the user's go-ahead and a Colab cap** (ticket 06a: `needs-triage`). This notebook was written in a CPU session and has never been run.

It runs the real attribution code on a Colab GPU **as a rehearsal**: calibration rows (`sequences-v2` calibration math) and the 20 rehearsal GSM8K test items only, for `gold-sft`, `online-grpo` and untuned. It never reads discovery, confirmation (held-out) rows or the ADR 0001 cohort. Every output is labelled `rehearsal-colab-*`, and production code refuses it (the record's `mode` is `rehearsal`).

Covering set (spec §Run tiers):
- calibration base traces;
- pass 1 and pass 2 with the Σ_j identity;
- real-model finite differences (3 adapters and 3 neurons per arm);
- no-op hook, single removal, group removal, full `down_proj` removal, natural 64-term read and constant read;
- full vs edited-checkpoint generation on 2 items;
- a corrupted and an interrupted unit;
- a token-budget probe from 2k upward with singleton and longest examples;
- online upload, fresh readback and a clean-root restore of this partial output.

**Runtime:** GPU A100 recommended. **Secret:** `HF_TOKEN` in Colab *Secrets* with read/write access to the private artifacts dataset.

If a step fails, run the **last cell** anyway. It uploads the rehearsal evidence so the failure can be diagnosed. Afterwards, fix each failure with a regression test and record it in `docs/agents/experiment-lessons.md`.

## 1. Settings

Values marked `None` have no approved value yet. The record builder refuses to start until each is set (see the open questions on tickets 01, 04 and 11). Fill them only with values the user has given.

In [ ]:
BRANCH = "claude/bold-gauss-5qkzwi"
REPO_URL = "https://github.com/mahadikprasad15/spar-pilot.git"
WORK = "/content/attribution"                 # local Colab disk
RUN_TAG = "r1"                                 # bump for a fresh rehearsal (r2, r3, ...)
ARMS = ["gold-sft", "online-grpo"]             # rehearsal arm set (untuned is always included)

# Approved by the user 2026-11-14
RANDOM_ORDER_SEEDS = [20001, 20002, 20003]     # the 3 random 64-neuron orders (primary 3 baseline)
NULL_SEED = 20004                              # the 1,000 random orders behind the concentration null
EARLY_STOP_RATIO = 0.5                         # early stop: EOS before 0.5 x the untuned response to that question
TIMEOUTS_SECONDS = dict(copy=2700, export_attempt=3600, stop_attempt=120, writer_join=300)
MARKER_STALE_SECONDS = 900
# Rehearsal items: seed-42 draw outside ADR 0001 (pilot_eval.attribution_backend.rehearsal_draw)

# Still needed before running
COLAB_CAP_HOURS = None                         # the user's Colab cap for 06a (required)
FP32_PEAK_FLOPS = None                         # dense FP32 peak of the Colab GPU (A100: 19.5e12), for MFU

## 2. Clone the branch and install the pinned packages

Steps run as subprocesses with their output streamed into the cell (Colab hides a child's own stderr), so the kernel never imports torch.

In [ ]:
import os, subprocess, sys, json, pathlib
def sh(cmd, env=None, check=True):
    """Run a shell step and stream its stdout+stderr into the cell."""
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line, end="", flush=True)
    code = proc.wait()
    if check and code:
        raise RuntimeError(f"command failed ({code}): {cmd}")
    return subprocess.CompletedProcess(cmd, code)

os.makedirs(WORK, exist_ok=True)
REPO = f"{WORK}/spar-pilot"
if not os.path.isdir(REPO):
    sh(f"git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO}")
else:
    sh(f"git -C {REPO} fetch origin {BRANCH} && git -C {REPO} checkout {BRANCH} && git -C {REPO} pull --ff-only")
COMMIT = subprocess.run(f"git -C {REPO} rev-parse HEAD", shell=True, text=True, capture_output=True).stdout.strip()
print("commit", COMMIT)
# Colab's preinstalled torchvision/torchaudio break `import transformers` once torch is pinned (lesson r1).
sh(f"{sys.executable} -m pip uninstall -q -y torchvision torchaudio", check=False)
sh(f"{sys.executable} -m pip install -q torch==2.9.0 transformers==4.57.6 peft==0.18.0 accelerate==1.12.0 "
   "huggingface-hub==0.36.0 tokenizers==0.22.1 safetensors pyyaml pyarrow jinja2 nvidia-ml-py")
sh(f"{sys.executable} -c 'import torch, transformers, peft; print(torch.__version__, transformers.__version__, peft.__version__, torch.cuda.get_device_name(0))'")

## 3. Environment (secrets never printed)

The worker runs offline (`HF_HUB_OFFLINE=1`); uploads and readbacks run in a separate online child process (`OnlineHubProcess`).

In [ ]:
from google.colab import userdata
ENV = dict(os.environ)
ENV.update(HF_TOKEN=userdata.get("HF_TOKEN"), HF_HOME=f"{WORK}/hf", PYTHONUNBUFFERED="1",
           CUBLAS_WORKSPACE_CONFIG=":4096:8", PYTHONPATH=REPO, HF_HUB_ENABLE_HF_TRANSFER="0")
ONLINE = dict(ENV, HF_HUB_OFFLINE="0", TRANSFORMERS_OFFLINE="0")
WORKER = dict(ENV, HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
print("HF token set:", bool(ENV["HF_TOKEN"]))
PY = sys.executable

## 4. Restore the sealed preparation snapshot

The joint audit's `prep-v2`: inventory-v2, sequences-v2, the step-64 adapters, the tokenizer and the GSM8K files, all hash-verified.

In [ ]:
PREP = f"{WORK}/prep"
RECEIPT = f"{REPO}/.scratch/joint-neuron-adapter-audit/prep-v2.hf-upload.json"
if not os.path.isdir(f"{PREP}/runs"):
    sh(f"cd {REPO} && {PY} -m pilot_eval.neuron_audit_export restore --upload_receipt {RECEIPT} --clean_root {PREP}", env=ONLINE)
SEQ = f"{PREP}/runs/joint-neuron-adapter-audit/sequences/sequences-v2"
INV = f"{PREP}/runs/joint-neuron-adapter-audit/inventory/inventory-v2"
sh(f"cd {REPO} && {PY} -m pilot_eval.neuron_audit_sequences verify --run_dir {SEQ}", env=ENV)

## 5. Download the pinned base model (online) and check its file hashes

In [ ]:
inventory = json.loads(pathlib.Path(f"{INV}/inventory.json").read_text())
base = inventory["base"]
request = dict(model=base["model"], revision=base["model_revision"], expected=base["expected_file_sha256"])
pathlib.Path(f"{WORK}/model_request.json").write_text(json.dumps(request))
script = """
import hashlib, json, pathlib, sys
from huggingface_hub import snapshot_download
request = json.loads(pathlib.Path(sys.argv[1]).read_text())
path = snapshot_download(request["model"], revision=request["revision"])
bad = []
for name, digest in request["expected"].items():
    h = hashlib.sha256()
    with open(pathlib.Path(path) / name, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 24), b""):
            h.update(chunk)
    if h.hexdigest() != digest:
        bad.append(name)
print(json.dumps(dict(path=path, mismatched=bad)))
"""
out = subprocess.run([PY, "-c", script, f"{WORK}/model_request.json"], env=ONLINE, text=True, capture_output=True)
if out.returncode:
    print(out.stderr[-3000:]); raise RuntimeError("model download failed")
MODEL = json.loads(out.stdout.strip().splitlines()[-1]); print(MODEL)
assert not MODEL["mismatched"], "base model files differ from the pinned hashes"

## 5b. Locate the pilot response files (for Stage D's item-level comparison)

Lists the files in the private dataset whose path names the step-64 pilot runs: `pilot2-sft-fp32-l4-batch8-v1-step-64` (gold-sft) and `pilot4-grpo-fp32-v1-behaviour-v1` (online-grpo; filter its rows to step 64). Copy the exact paths into the launch record's `stage_d.pilot_results`. Their responses are re-scored with flexible v3, and stored scores that disagree are listed.

In [ ]:
script = """
from huggingface_hub import HfApi
files = HfApi().list_repo_files('Prasadmahadik/spar-sampling-sft-artifacts', repo_type='dataset')
for name in files:
    if 'pilot2-sft-fp32-l4-batch8-v1-step-64' in name or 'pilot4-grpo-fp32-v1-behaviour-v1' in name:
        print(name)
"""
pathlib.Path(f"{WORK}/list_pilot_files.py").write_text(script)
sh(f"{PY} {WORK}/list_pilot_files.py", env=ONLINE, check=False)

## 6. Rehearsal record (rehearsal only; never a launch record)

The tolerances, sweeps and statistics are copied unchanged from `candidate-plan-v1.yaml` (all still `[proposed]`). In this rehearsal they are only used to *exercise* the checks; nothing here approves them. The record is sealed against this commit.

In [ ]:
missing = [n for n, v in dict(COLAB_CAP_HOURS=COLAB_CAP_HOURS, FP32_PEAK_FLOPS=FP32_PEAK_FLOPS).items() if v is None]
assert not missing, f"set these in cell 1 first (user decisions): {missing}"
manifest = json.loads(pathlib.Path(f"{SEQ}/manifest.json").read_text())
adapter_dirs = {arm: f"{PREP}/{rec['checkpoints']['64']['path']}" for arm, rec in inventory["arms"].items()
                if rec["checkpoints"].get("64") and arm in ARMS}
cap = int(COLAB_CAP_HOURS * 3600)
record = dict(
    schema="adapter-neuron-attribution/launch-record-v1", run_id=f"rehearsal-colab-{RUN_TAG}", mode="rehearsal",
    rehearsal_label=f"rehearsal-colab-{RUN_TAG}", code_commit=COMMIT, pod_id="colab-no-pod",
    pod_stop_commands=["true {id}"], expected_arms=ARMS, volume_identity="colab-local",
    export_repo="Prasadmahadik/spar-sampling-sft-artifacts", hourly_cost_usd=0.0,
    caps_seconds=dict(total=cap, preflight=cap, qualification=cap, base_traces=cap, attribution_and_freeze=cap,
                      stage_c=cap, stage_d=cap, report=cap),
    tolerances=dict(finite_difference=dict(epsilon=1.0e-2, median_rel_err_max=0.05, max_rel_err_max=0.20),
                    neuron_sum_identity=1.0e-4, batch_invariance=1.0e-4,
                    resolution_floor="10x measured batch-invariance discrepancy"),
    minimums=dict(discovery=120, heldout=60),
    selection=dict(top_group=10, top_singles=5, tolerance=0.20, seed=20000, random_seeds=RANDOM_ORDER_SEEDS,
                   read_size=64),
    statistics=dict(resamples=10000, seed=1234, alpha=0.05, margin=0.25, sparse_lower=0.70, dense_upper=0.30,
                    gate_singles_min=4, null_orders=1000, null_seed=NULL_SEED),
    generation_batch_rule=dict(candidates=[1, 8, 16, 32, 64], items=20, max_flips=1, max_stop_changes=1),
    token_budget_sweep=[2048, 4096, 8192, 16384, 32768, 65536], watchdog_margin_seconds=1800,
    marker_stale_seconds=MARKER_STALE_SECONDS, timeouts_seconds=TIMEOUTS_SECONDS, compute=dict(fp32_peak_flops=FP32_PEAK_FLOPS),
    stage_d=dict(early_stop_rule=dict(kind="fraction-of-untuned-same-item", ratio=EARLY_STOP_RATIO),
                 pilot_results={}),
    qualification=dict(calibration_minimum=10, fd_arms=ARMS, fd_rows=4, fd_adapters=3, fd_neurons=3,
                       head_chunk=1024, memory_fraction=0.80),
    inputs=dict(artifact_root=PREP, inventory_dir=INV, sequences_dir=SEQ, model_dir=MODEL["path"],
                tokenizer_dir=f"{PREP}/inputs/models/{base['tokenizer_revision']}",
                gsm8k_test_parquet=manifest["sources"]["gsm8k"]["test_parquet"], adapter_dirs=adapter_dirs,
                max_new_tokens=1024, work_dir=f"{WORK}/local"))
for d in ["launch", "local", "volume"]:
    os.makedirs(f"{WORK}/{d}", exist_ok=True)
pathlib.Path(f"{WORK}/volume/volume_identity.json").write_text(json.dumps(dict(identity="colab-local")))
RECORD = f"{WORK}/launch/rehearsal_record.json"
pathlib.Path(f"{WORK}/launch/draft.json").write_text(json.dumps(record, indent=2))
sh(f"cd {REPO} && {PY} -c \"import json; from pilot_eval.attribution_chain import seal_launch_record, load_launch_record; "
   f"seal_launch_record(json.load(open('{WORK}/launch/draft.json')), '{RECORD}'); "
   f"print(load_launch_record('{RECORD}', code_commit='{COMMIT}')['run_id'])\"", env=ENV)

## 7. Run the covering set

`python -m pilot_eval.attribution_rehearsal` runs every step and records each failure with its traceback, then continues with the steps that don't depend on it. It writes a sealed summary, uploads with fresh readback and restores into a clean root. The exit code is 0 only if every step and the export succeeded.

In [ ]:
CLEAN = f"{WORK}/clean-{RUN_TAG}"
result = sh(f"cd {REPO} && {PY} -m pilot_eval.attribution_rehearsal --record {RECORD} --work {WORK} --clean_root {CLEAN}",
            env=WORKER, check=False)
print("exit code", result.returncode)

## 8. Summary and failures

In [ ]:
from IPython.display import Markdown, display
summary_dir = pathlib.Path(f"{WORK}/local/attribution/summary")
if (summary_dir / "run_summary.md").is_file():
    display(Markdown((summary_dir / "run_summary.md").read_text()))
    summary = json.loads((summary_dir / "run_summary.json").read_text())
    tracebacks = json.loads(pathlib.Path(f"{WORK}/local/attribution/rehearsal/tracebacks/tracebacks.json").read_text())
    for name in summary["failed"]:
        print("=" * 20, name); print(tracebacks.get(name))
    print(json.dumps({k: v.get("result") for k, v in summary["steps"].items()
                      if k.startswith(("token_budget", "finite_differences"))}, indent=2, default=str)[:6000])
else:
    print("no summary was written; see the cell output above")

## 9. Evidence upload (run this even after a failure)

Uploads whatever exists, including unsealed files labelled as evidence, under `rehearsal-colab-<tag>-evidence`. The receipt is printed here, since Colab hides a child process's own output (lesson r1).

In [ ]:
script = f"""
import json
from pilot_eval import neuron_audit_export as ex
snap = ex.snapshot('{WORK}/local', ['attribution'], 'rehearsal-colab-{RUN_TAG}-evidence', include_unsealed_evidence=True)
receipt = ex.upload('{WORK}/local/' + snap['snapshot_path'], '{WORK}/local', repo='Prasadmahadik/spar-sampling-sft-artifacts',
                    temp_dir='{WORK}/tmp')
print('EVIDENCE RECEIPT', json.dumps(dict(revision=receipt['revision'], completeness=receipt['completeness'],
                                          files=receipt['file_count'])))
"""
os.makedirs(f"{WORK}/tmp", exist_ok=True)
pathlib.Path(f"{WORK}/evidence_upload.py").write_text(script)
sh(f"cd {REPO} && {PY} {WORK}/evidence_upload.py", env=ONLINE, check=False)